In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from anthropic import Anthropic
ai_client = Anthropic()

In [3]:
def llm(prompt):
    response = ai_client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=1024,
        messages=[{"role": "user", "content": prompt}]
    )
    return response.content[0].text

In [4]:
llm('hey whats up')

"Hey! Not much, just here and ready to chat. What's going on with you?"

In [5]:
question = "I just discovered the course. Can I join now?"
answer = llm(question)
print(answer)

I'd be happy to help! However, I don't have information about which specific course you're referring to or its enrollment details.

To give you accurate information, could you tell me:

- **Which course** are you asking about?
- **Where is it offered?** (e.g., a specific platform, school, or organization)
- **When does it start/what's the current status?**

With those details, I can help you find out about enrollment deadlines, requirements, and how to join!


In [6]:
context = """
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we're still accepting submissions.

Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

What is the video/zoom link to the stream for the "Office Hours" or live/workshop sessions?
The zoom link is only published to instructors/presenters/TAs. Students participate via YouTube Live and submit questions to Slido.

Cloud alternatives with GPU
Check the quota and reset cycle carefully. Potential options include Google Colab, Kaggle, Databricks.
"""

In [7]:
prompt = f"""
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

Question:
{question}

Context:
{context}
"""

In [8]:
answer = llm(prompt)
print(answer)

# Can You Join Now?

Yes, you can join the course now! However, there's an important condition:

**If you want to receive a certificate**, you'll need to submit your project while the course is still accepting submissions.

You can start learning and submitting homework immediately without needing to wait for any confirmation email. Registration is optional and is mainly used to gauge interest—your submissions won't be checked against a registered list.


In [9]:
# fetch the data from the FAQ that we'll use as an example. 
# this is already saved as a .json for convenience.
# in reality, most data you'll have to scrape and prepare for use.

import requests

docs_url = "https://datatalks.club/faq/json/courses.json"
response = requests.get(docs_url)
courses_raw = response.json()

In [10]:
# preview what courses_raw looks like
print(type(courses_raw))
print(len(courses_raw))
courses_raw[0]

<class 'list'>
6


{'course': 'data-engineering-zoomcamp',
 'course_name': 'Data Engineering Zoomcamp',
 'path': '/json/data-engineering-zoomcamp.json',
 'questions_count': 402}

In [11]:
# Now fetch the FAQ from all of DataTalkClub's courses:
documents = []
url_prefix = "https://datatalks.club/faq"

for course in courses_raw:
    course_url = f"""{url_prefix}{course["path"]}""" # builds URL for each course's FAQ

    course_response = requests.get(course_url) # fetches the FAQ data
    course_response.raise_for_status() # raises error if request fails
    course_data = course_response.json() #parses response into Python

    documents.extend(course_data) # adds all the docs from the course into the master 'documents' list

len(documents) # shows the total # of FAQ entries across all courses

1342

In [12]:
# preview a document
documents[0]

{'id': '9e508f2212',
 'course': 'data-engineering-zoomcamp',
 'section': 'General Course-Related Questions',
 'question': 'Course: When does the course start?',
 'answer': "A new cohort runs roughly January–April every year. For the current cohort's exact start date and registration link, check the [course repo README](https://github.com/DataTalksClub/data-engineering-zoomcamp).\n\n- Register via the link in the course repo before the cohort starts.\n- Join the [course Telegram channel](https://t.me/dezoomcamp) for announcements.\n- Join DataTalks.Club's [Slack](https://datatalks.club/docs/general/slack/) and the `#course-data-engineering` channel."}

In [13]:
# Use minsearch and define the Index fields
# keyword is for filtering

from minsearch import Index

index = Index(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"]
)

index.fit(documents) 
# this is the "loading" step: loops through documents, tokenizes text fields, 
# builds a lookup table of the keyword, and stores this in memory for retrieval
# 'fit' like fit a model on data. Here its fitting an index on documents

In [14]:
# Try a search using the question from before:

question = "I just discovered the course. Can I join now?"

search_results = index.search(
    question,
    boost_dict={"question": 2.0, "section": 0.5}, #question is 2x more important than the other text fields and section is less important.
    filter_dict={"course": "llm-zoomcamp"}, #filter only by courses that are in the llm-zoomcamp
    num_results=5 #limit to 5 results 
)

search_results #lists the search results 

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [15]:
# To view just the questions text from "search_results":
[doc["question"] for doc in search_results]

['I just discovered the course. Can I still join?',
 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
 'When will the course be offered next?',
 'I missed the first homework - can I still get a certificate?']

In [16]:
# Wrap 'search' in a search function - the first component of the RAG pipeline:

def search(question, course="llm-zoomcamp"):
    boost_dict={"question": 2.0, "section": 0.5} #question is 2x more important than the other text fields and section is less important.
    filter_dict={"course": course} # course is defined as llm-zc above as a default

    return index.search(
        question,
        boost_dict=boost_dict,
        filter_dict=filter_dict,
        num_results=5 #limit to 5 results 
    )

In [17]:
search_results = search(question)

In [18]:
search_results

[{'id': '74eb249bbf',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'I just discovered the course. Can I still join?',
  'answer': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'},
 {'id': '977bf7786c',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?',
  'answer': "You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date."},
 {'id': '69d122f12e',
  'course': 'llm-zoomcamp',
  'section': 'General Course-Related Questions',
  'question': 'Certificate: Can I follow the course in a self-paced mode and get a certificate?',
  'answer': 'No, you c

In [19]:
# Now Retrieval is complete
# Time to build the Prompt (use what we did above)
# When build, split into 2 parts for AI systems:
# 1. Instructions ()
# 2. User prompt (changes with every request)

# INSTRUCTIONS FOR PROMPT
INSTRUCTIONS = """
Your task is to answer questions from the course participants
based on the provided context.

Use the context to find relevant information and provide accurate
answers. If the answer is not found in the context,
respond with "I don't know."

"""

# USER PROMPT
USER_PROMPT_TEMPLATE = '''
Question:
{question}

Context:
{context}
'''

In [20]:
# Build the Context
# 'context' is a formatted string with all the search results

def build_context(search_results):
    lines = [] #empty list holds each line of text before joining them 

    for doc in search_results: #append the four text items to 'lines' list
        lines.append(doc["section"])
        lines.append("Q: " + doc["question"])
        lines.append("A: " + doc["answer"])
        lines.append("") #create a blank line between docs

    return "\n".join(lines).strip() #joins all the lines into one string with newlines (\n) between them. Strips leading/trailing whitespace.

In [21]:
# See an example of output
# This format is easier for the LLM to read
build_context(search_results)

'General Course-Related Questions\nQ: I just discovered the course. Can I still join?\nA: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.\n\nGeneral Course-Related Questions\nQ: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?\nA: You don\'t need it. You\'re accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.\n\nGeneral Course-Related Questions\nQ: Certificate: Can I follow the course in a self-paced mode and get a certificate?\nA: No, you can only get a certificate if you finish the course with a "live" cohort.\n\nWe don\'t award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project.\n\nYou can only peer-review projects at the time

In [22]:
# Build the Prompt
# combine the question with the context into the user prompt:

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = USER_PROMPT_TEMPLATE.format( #replaces question and context with actual values
        question=question,
        context=context
    )
    return prompt.strip() #trims extra whitespace from edges of the final string

# This is the core of RAG — you're not just sending the question 
# to the LLM, you're sending the question plus the relevant 
# retrieved documents so the model can ground its answer in actual 
# course FAQ content rather than making things up.

In [23]:
# Test the prompt using the question and search_results already defined earlier.

prompt = build_prompt(question, search_results)

print(prompt)

Question:
I just discovered the course. Can I join now?

Context:
General Course-Related Questions
Q: I just discovered the course. Can I still join?
A: Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

General Course-Related Questions
Q: Course: I have registered for the LLM Zoomcamp. When can I expect to receive the confirmation email?
A: You don't need it. You're accepted. You can also just start learning and submitting homework (while the form is open) without registering. It is not checked against any registered list. Registration is just to gauge interest before the start date.

General Course-Related Questions
Q: Certificate: Can I follow the course in a self-paced mode and get a certificate?
A: No, you can only get a certificate if you finish the course with a "live" cohort.

We don't award certificates for the self-paced mode. The reason is you need to peer-review 3 capstone(s) after submitting your project

In [24]:
# Earlier we defined 'llm' as a function with input of a prompt
# that would provide a response. Use that response code here:
#
#def llm(prompt):
#    response = ai_client.messages.create(
#        model="claude-haiku-4-5-20251001",
#        max_tokens=1024,
#        messages=[{"role": "user", "content": prompt}]
#    )
#    return response.content[0].text

response = ai_client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    messages = [{"role": "user", "content": prompt}]
)

In [25]:
# Explore the response
# Response is a Pydantic object. To see the output:
response.content[0]

# "response.output[0]" (OpenAI) = "response.content[0]" (Anthropic)
# "response.output_text" (OpenAI) = "response.content[0].text" (Anthropic)


TextBlock(citations=None, text='# Yes, you can join now!\n\nBased on the course information:\n\n✅ **You can start learning immediately** - No registration confirmation email is needed. You\'re accepted and can begin right away.\n\n✅ **You can submit homework** - While the submission form is still open, you can complete and submit assignments.\n\n⚠️ **Important for certificates** - If you want to receive a certificate, you\'ll need to:\n- Submit your capstone project while submissions are still being accepted\n- Complete the peer-review process (reviewing 3 capstone projects from other students)\n\n**Note:** Certificates are only available for students completing the course with a "live" cohort, not in self-paced mode. The peer-review process can only happen while the course is actively running.\n\nIf you\'re interested in joining the next cohort, the course will be offered again in **Summer 2025**.', type='text')

In [26]:
# The first one above is the message
# The message has a content list and the text is in the first item of that list

response.content[0].text

'# Yes, you can join now!\n\nBased on the course information:\n\n✅ **You can start learning immediately** - No registration confirmation email is needed. You\'re accepted and can begin right away.\n\n✅ **You can submit homework** - While the submission form is still open, you can complete and submit assignments.\n\n⚠️ **Important for certificates** - If you want to receive a certificate, you\'ll need to:\n- Submit your capstone project while submissions are still being accepted\n- Complete the peer-review process (reviewing 3 capstone projects from other students)\n\n**Note:** Certificates are only available for students completing the course with a "live" cohort, not in self-paced mode. The peer-review process can only happen while the course is actively running.\n\nIf you\'re interested in joining the next cohort, the course will be offered again in **Summer 2025**.'

In [27]:
# See how much tokens you consumed
# Refer back to course page if you want calculation for tokens

response.usage

Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=373, output_tokens=200, output_tokens_details=None, server_tool_use=None, service_tier='standard')

In [28]:
# Message History
# developer = system-level instructions
# user = actual prompt with question + context
# this is set up to separate instructions from user prompt which changes with every request

message_history = [
    {"role": "developer", "content": INSTRUCTIONS},
    {"role": "user", "content": prompt}
]

response = ai_client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    messages = [{"role": "user", "content": prompt}]
)

In [29]:
# Update the LLM function to take both instructions and user prompt

# BEFORE:
#def llm(prompt):
#    response = ai_client.messages.create(
#        model="claude-haiku-4-5-20251001",
#        max_tokens=1024,
#        messages=[{"role": "user", "content": prompt}]
#    )
#    return response.content[0].text

# UPDATED:
def llm(instructions, user_prompt, model="claude-haiku-4-5-20251001"):
    # message history separating instructions from user prompt
    message_history = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": user_prompt}
    ]

    # define response parameters
    response = ai_client.messages.create(
    model = "claude-haiku-4-5-20251001",
    max_tokens = 1024,
    messages = [{"role": "user", "content": prompt}]
    )

    # return the response message text (at [0])
    return response.content[0].text

# REMINDERS: Code defined above is
# prompt = build_prompt(question, search_results)

In [30]:
# FULL RAG
# with search, prompt, and LLM ready we put them together in RAG:

def rag(query, model="claude-haiku-4-5-20251001"):
    search_results = search(query)
    prompt = build_prompt(query, search_results)
    answer = llm(INSTRUCTIONS, prompt, model=model)
    return answer

In [31]:
# Test RAG:

answer = rag("I just discovered the course. Can I join now?")

print(answer)

# Yes, you can join now!

Based on the course information:

✅ **You can start learning immediately** - You don't need to wait for confirmation or be on a registered list. Just begin with the course materials.

✅ **You can submit homework** - While the submission form is still open, you're welcome to start and submit assignments.

⚠️ **Important for certificates** - If you want to receive a certificate, you'll need to:
1. **Submit your capstone project while submissions are still being accepted**
2. Complete the course as part of a "live" cohort (not self-paced)
3. Peer-review 3 capstone projects after submitting your own

Since the next cohort is scheduled for **Summer 2025**, make sure you're joining during an active course run to be eligible for certification and peer-review participation.


In [32]:
# Test again:
rag("How do I get a certificate?")

"# Yes, you can join now!\n\nBased on the course guidelines:\n\n✅ **You can start learning immediately** - No confirmation email or registration verification is required. You can begin the course content right away.\n\n⚠️ **Important for certificates:** If you want to receive a certificate, you'll need to:\n1. **Submit your capstone project while submissions are still open**\n2. **Complete peer reviews** - You must peer-review 3 capstone projects from other students (this happens during the live course cohort)\n\n**Note:** Certificates are only awarded for courses completed with a live cohort, not for self-paced learning, since peer-review is a required component.\n\nIf you're interested in getting a certificate, the next cohort runs in **Summer 2025**."

In [33]:
# Test again, but using print function. This makes the format easier to read...
answer = rag("How do I get a certificate?")
print(answer)

# Yes, you can join now!

Based on the course information:

✅ **You can start learning immediately** - No registration confirmation is required, and you can begin the course right away.

✅ **You can submit assignments** - You're able to submit homework and projects while the submission form is still open.

⚠️ **Important for certificates** - If you want to receive a certificate, you'll need to:
- Submit your capstone project while submissions are still being accepted
- Complete peer reviews of other students' projects (this only happens during the live cohort period)

**Note:** The course is currently running as a live cohort. If you're interested in future offerings, the next course is scheduled for **Summer 2025**.
